In [1]:
%pip install -U trl

Note: you may need to restart the kernel to use updated packages.


In [2]:
from importlib import reload

from Trainers.trainer_classifier import ClassifierTrainer, ClassifierTrainingConfig
from Trainers.trainer_ppo import PPOTrainingConfig, PolicyPPOTrainer
from Models.model_policy import PolicyModel
from Models.model_value import ValueModel
from Models.model_reward import RewardModel
from Models.model_classifier import Classifier
import Datasets.dataset_request as dataset_request


dataset_request = reload(dataset_request)
RequestDataset = dataset_request.RequestDataset

c:\Users\misha\anaconda3\envs\torch310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import logging

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    force=True,
)

In [4]:
dataset = RequestDataset.load("human_requests_hh-rlhf.pt", "Qwen/Qwen3-0.6B")
dataset.truncate(0, 5)

2026-09-01 18:14:18,544 | INFO | Datasets.dataset_request | PATH: human_requests_hh-rlhf.pt
2026-09-01 18:14:18,696 | DEBUG | Datasets.dataset_request | Expecting list: <class 'list'>
2026-09-01 18:14:18,707 | DEBUG | httpcore.connection | connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-09-01 18:14:18,726 | DEBUG | httpcore.connection | connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x000001CBC91F46D0>
2026-09-01 18:14:18,727 | DEBUG | httpcore.connection | start_tls.started ssl_context=<ssl.SSLContext object at 0x000001CBCB17AE40> server_hostname='huggingface.co' timeout=10
2026-09-01 18:14:18,736 | DEBUG | httpcore.connection | start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x000001CBC91F46A0>
2026-09-01 18:14:18,736 | DEBUG | httpcore.http11 | send_request_headers.started request=<Request [b'HEAD']>
2026-09-01 18:14:18,737 | DEBUG | httpcore.http11 | send_request

In [5]:
config = PPOTrainingConfig(
    output_dir="outputs/ppo_policy"
)
policy = PolicyModel("Qwen/Qwen3-0.6B")
value = ValueModel("Qwen/Qwen3-0.6B")
reward_model = RewardModel("Skywork/Skywork-Reward-V2-Qwen3-0.6B", "proxy")
judge = RewardModel("Skywork/Skywork-Reward-V2-Qwen3-4B", "judge")

2026-09-01 18:14:20,058 | DEBUG | httpcore.http11 | send_request_headers.started request=<Request [b'HEAD']>
2026-09-01 18:14:20,059 | DEBUG | httpcore.http11 | send_request_headers.complete
2026-09-01 18:14:20,059 | DEBUG | httpcore.http11 | send_request_body.started request=<Request [b'HEAD']>
2026-09-01 18:14:20,060 | DEBUG | httpcore.http11 | send_request_body.complete
2026-09-01 18:14:20,061 | DEBUG | httpcore.http11 | receive_response_headers.started request=<Request [b'HEAD']>
2026-09-01 18:14:20,205 | DEBUG | httpcore.http11 | receive_response_headers.complete return_value=(b'HTTP/1.1', 307, b'Temporary Redirect', [(b'Content-Type', b'text/plain; charset=utf-8'), (b'Content-Length', b'234'), (b'Connection', b'keep-alive'), (b'Date', b'Tue, 01 Sep 2026 15:14:20 GMT'), (b'Location', b'/api/resolve-cache/models/Qwen/Qwen3-0.6B/c1899de289a04d12100db370d81485cdf75e47ca/config.json?%2FQwen%2FQwen3-0.6B%2Fresolve%2Fmain%2Fconfig.json=&etag=%22f5c3703b78ae2a478ae15b247e9f855e0ce2107b%2

In [6]:
policy.generate_new_dataset(dataset, 4)

2026-09-01 18:14:43,250 | DEBUG | httpcore.connection | close.started
2026-09-01 18:14:43,278 | DEBUG | httpcore.connection | close.complete
2026-09-01 18:14:43,279 | DEBUG | httpcore.connection | connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-09-01 18:14:43,326 | DEBUG | httpcore.connection | connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x000001CC371ECE20>
2026-09-01 18:14:43,328 | DEBUG | httpcore.connection | start_tls.started ssl_context=<ssl.SSLContext object at 0x000001CBCB17AE40> server_hostname='huggingface.co' timeout=10
2026-09-01 18:14:43,364 | DEBUG | httpcore.connection | start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x000001CC371ECA30>
2026-09-01 18:14:43,365 | DEBUG | httpcore.http11 | send_request_headers.started request=<Request [b'HEAD']>
2026-09-01 18:14:43,367 | DEBUG | httpcore.http11 | send_request_headers.complete
2026-09-01 18:14:43,368 |

RequestDataset(size=5)

In [7]:
reward_model.init_normalization(policy)
judge.init_normalization(policy)

reward_model.score_policy(policy)
judge.score_policy(policy)

2026-09-01 18:16:27,874 | INFO | Models.model_reward | proxy: Starting to calculate the score
2026-09-01 18:16:33,268 | DEBUG | Models.model_reward | proxy: Reward scores: [-6.90625, -2.515625, -3.078125, 2.015625, -7.0]
2026-09-01 18:16:33,271 | DEBUG | Models.model_reward | proxy: Combined reward scores: [-6.90625, -2.515625, -3.078125, 2.015625, -7.0]
2026-09-01 18:16:33,291 | INFO | Models.model_reward | judge: Starting to calculate the score
2026-09-01 18:17:02,878 | DEBUG | Models.model_reward | judge: Reward scores: [-5.90625, -3.015625, -1.1171875, 9.875, -7.75]
2026-09-01 18:17:02,879 | DEBUG | Models.model_reward | judge: Combined reward scores: [-5.90625, -3.015625, -1.1171875, 9.875, -7.75]
2026-09-01 18:17:02,880 | INFO | Models.model_reward | proxy: Starting to calculate the score
2026-09-01 18:17:07,421 | DEBUG | Models.model_reward | proxy: Reward scores: [-6.90625, -2.515625, -3.078125, 2.015625, -7.0]
2026-09-01 18:17:07,422 | DEBUG | Models.model_reward | proxy: Comb

In [8]:
classfier_dataset = policy.generate_dataset_classifier(2)
train, test = classfier_dataset.split()

In [9]:
classifier_config = ClassifierTrainingConfig(output_dir="outputs/classifiers")
classifier = Classifier("1", "Qwen/Qwen3-0.6B")

classifier_trainer = ClassifierTrainer(classifier, classifier_config)
classifier_trainer.train(train, test)

2026-09-01 18:17:40,498 | DEBUG | httpcore.http11 | send_request_headers.started request=<Request [b'HEAD']>
2026-09-01 18:17:40,500 | DEBUG | httpcore.http11 | send_request_headers.complete
2026-09-01 18:17:40,501 | DEBUG | httpcore.http11 | send_request_body.started request=<Request [b'HEAD']>
2026-09-01 18:17:40,502 | DEBUG | httpcore.http11 | send_request_body.complete
2026-09-01 18:17:40,502 | DEBUG | httpcore.http11 | receive_response_headers.started request=<Request [b'HEAD']>
2026-09-01 18:17:40,657 | DEBUG | httpcore.http11 | receive_response_headers.complete return_value=(b'HTTP/1.1', 307, b'Temporary Redirect', [(b'Content-Type', b'text/plain; charset=utf-8'), (b'Content-Length', b'234'), (b'Connection', b'keep-alive'), (b'Date', b'Tue, 01 Sep 2026 15:17:41 GMT'), (b'Location', b'/api/resolve-cache/models/Qwen/Qwen3-0.6B/c1899de289a04d12100db370d81485cdf75e47ca/config.json?%2FQwen%2FQwen3-0.6B%2Fresolve%2Fmain%2Fconfig.json=&etag=%22f5c3703b78ae2a478ae15b247e9f855e0ce2107b%2

OutOfMemoryError: CUDA out of memory. Tried to allocate 64.00 MiB. GPU 0 has a total capacity of 8.00 GiB of which 0 bytes is free. Of the allocated memory 14.49 GiB is allocated by PyTorch, and 25.80 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)